In [ ]:
import sys

sys.path.insert(0, ".")
import gc
import time

import base
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Audio

torch.manual_seed(base.SEED)
np.random.seed(base.SEED)
torch.set_num_threads(max(1, torch.get_num_threads()))

## Case 1: English accent classification with Wav2Vec 2.0
Common Voice (Kaggle, public): `us` / `australia` / `england`. Only the CSVs and a balanced sample of
clips are fetched. `wav2vec2-base` (95M parameters) is used as a **frozen feature extractor**: every transformer layer is
mean-pooled, the best layer is picked by cross-validation and a logistic-regression head is trained on it (minutes on CPU).
Full fine-tuning (`FINETUNE = True`) is the GPU version of this case.

In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel

In [ ]:
p2c1 = base.Case(2, 1)
SR, SECONDS = 16000, 3
ACCENTS = ["us", "australia", "england"]
N_TRAIN, N_TEST = 400, 80  # clips per accent
BACKBONE = "facebook/wav2vec2-base"
FINETUNE = False  # True = fine-tune the whole backbone (GPU recommended)
HANDLE = "mozillaorg/common-voice"


def sample_clips(split, n):
    (csv,) = base.fetch_kaggle_files(HANDLE, [f"cv-valid-{split}.csv"], p2c1.data)
    df = pd.read_csv(csv, usecols=["filename", "accent"])
    df = (
        df[df.accent.isin(ACCENTS)]
        .groupby("accent")
        .sample(n, random_state=base.SEED)
        .reset_index(drop=True)
    )
    df["path"] = base.fetch_kaggle_files(
        HANDLE, [f"cv-valid-{split}/{f}" for f in df.filename], p2c1.data
    )
    df["label"] = df.accent.map(ACCENTS.index)
    return df


def to_wave(path):
    """16 kHz, fixed 3 s (trim / zero-pad), zero-mean unit-variance (what the wav2vec2 feature extractor does)."""
    y = base.load_audio(path, SR)[: SR * SECONDS]
    y = np.pad(y, (0, SR * SECONDS - len(y)))
    return ((y - y.mean()) / (y.std() + 1e-7)).astype("float32")


train, test = sample_clips("train", N_TRAIN), sample_clips("test", N_TEST)
W = {
    k: np.stack([to_wave(p) for p in d.path])
    for k, d in {"train": train, "test": test}.items()
}
print({k: v.shape for k, v in W.items()})

In [ ]:
# Layer-wise mean-pooled embeddings (cached): (clips, 13 layers, 768)
cache = p2c1.data / f"embeddings_{N_TRAIN}_{N_TEST}.npz"
if cache.exists():
    E = dict(np.load(cache))
else:
    enc = AutoModel.from_pretrained(BACKBONE, cache_dir=p2c1.data).eval()

    @torch.inference_mode()
    def embed(waves, bs=4):  # small batches keep peak RAM low
        out = []
        for i in range(0, len(waves), bs):
            hs = enc(
                torch.tensor(waves[i : i + bs]), output_hidden_states=True
            ).hidden_states
            out.append(torch.stack([h.mean(1) for h in hs], 1).numpy())
        return np.concatenate(out)

    t0 = time.perf_counter()
    E = {k: embed(v) for k, v in W.items()}
    print(f"embedding time: {time.perf_counter() - t0:.0f}s")
    np.savez_compressed(cache, **E)
    del enc
    gc.collect()

Which layer carries the accent information? (5-fold CV accuracy on the training clips)

In [ ]:
y_tr, y_te = train.label.values, test.label.values
head = lambda C=1.0: make_pipeline(
    StandardScaler(), LogisticRegression(C=C, max_iter=3000, random_state=base.SEED)
)
cv = StratifiedKFold(5, shuffle=True, random_state=base.SEED)
layer_acc = [
    cross_val_score(head(), E["train"][:, l], y_tr, cv=cv, n_jobs=-1).mean()
    for l in range(E["train"].shape[1])
]

best_layer = int(np.argmax(layer_acc))

plt.figure(figsize=(6, 3))
plt.plot(layer_acc, "o-")
plt.axhline(1 / len(ACCENTS), ls="--", c="gray", label="chance")
(
    plt.xlabel("transformer layer (0 = CNN output)"),
    plt.ylabel("CV accuracy"),
    plt.legend(),
    plt.title(f"Best layer: {best_layer}"),
)

p2c1.save_fig("layer_accuracy")

In [ ]:
search = GridSearchCV(
    head(), {"logisticregression__C": [0.001, 0.01, 0.1, 1]}, cv=cv, n_jobs=-1
).fit(E["train"][:, best_layer], y_tr)
clf = search.best_estimator_
proba = clf.predict_proba(E["test"][:, best_layer])
pred = proba.argmax(1)

metrics = {
    "layer": best_layer,
    "C": search.best_params_["logisticregression__C"],
    "cv_acc": search.best_score_,
    "test_acc": accuracy_score(y_te, pred),
    "test_f1_macro": f1_score(y_te, pred, average="macro"),
    "test_roc_auc": roc_auc_score(y_te, proba, multi_class="ovr"),
}

print({k: round(v, 3) if isinstance(v, float) else v for k, v in metrics.items()})
print(classification_report(y_te, pred, target_names=ACCENTS))
plt.figure(figsize=(4, 3.5))
base.plot_confusion(y_te, pred, ACCENTS, f"Accent head, layer {best_layer} (test)")

p2c1.save_fig("confusion_test")

# Model = frozen wav2vec2 (reference only) + our trained head
joblib.dump(
    {
        "head": clf,
        "layer": best_layer,
        "backbone": BACKBONE,
        "accents": ACCENTS,
        "sr": SR,
        "seconds": SECONDS,
    },
    p2c1.model / "accent_head.joblib",
)

p2c1.save_json(
    "reference",
    {
        "backbone": BACKBONE,
        "note": "frozen pretrained backbone, downloaded from HuggingFace; not stored here",
        **metrics,
    },
    model=True,
)

GPU version: fine-tune the full backbone (`FINETUNE = True`). Not needed for the results above.

In [ ]:
if FINETUNE:
    from sklearn.model_selection import train_test_split
    from transformers import Wav2Vec2ForSequenceClassification

    class Wav2VecClassifier(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.m = Wav2Vec2ForSequenceClassification.from_pretrained(
                BACKBONE, num_labels=len(ACCENTS), cache_dir=p2c1.data
            )
            self.m.freeze_feature_encoder()

        def forward(self, x):
            return self.m(input_values=x).logits

    i_tr, i_va = train_test_split(
        np.arange(len(y_tr)), test_size=0.1, stratify=y_tr, random_state=base.SEED
    )

    dl = lambda x, y, shuffle: torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(x), torch.tensor(y)),
        batch_size=8,
        shuffle=shuffle,
    )

    net, hist = base.train_classifier(
        Wav2VecClassifier(),
        dl(W["train"][i_tr], y_tr[i_tr], True),
        dl(W["train"][i_va], y_tr[i_va], False),
        epochs=5,
        lr=3e-5,
        patience=2,
    )

    ft_pred = (
        base.predict_torch(net, torch.tensor(W["test"])).argmax(1)
        if len(W["test"]) < 64
        else np.concatenate(
            [
                base.predict_torch(net, torch.tensor(W["test"][i : i + 16])).argmax(1)
                for i in range(0, len(y_te), 16)
            ]
        )
    )

    print("fine-tuned test accuracy:", accuracy_score(y_te, ft_pred))
    base.plot_history(hist, p2c1, "finetune_curves")
    net.m.save_pretrained(p2c1.model / "finetuned")  # our own trained weights (~360 MB)

Inference example: one test clip per accent

In [ ]:
bundle = joblib.load(p2c1.model / "accent_head.joblib")
backbone = AutoModel.from_pretrained(bundle["backbone"], cache_dir=p2c1.data).eval()


@torch.no_grad()
def predict_accent(path):
    hs = backbone(
        torch.tensor(to_wave(path))[None], output_hidden_states=True
    ).hidden_states[bundle["layer"]]
    p = bundle["head"].predict_proba(hs.mean(1).numpy())[0]
    return bundle["accents"][int(p.argmax())], {
        a: round(float(x), 3) for a, x in zip(bundle["accents"], p)
    }


for _, r in test.groupby("accent").head(1).iterrows():
    print(f"true: {r.accent:10s} ->", predict_accent(r.path))
del backbone

gc.collect()  # free RAM for the next case
Audio(to_wave(r.path), rate=SR)

## Case 2: Automatic speech recognition with Whisper (FLEURS, Indonesian)
Based on the OpenAI Whisper multilingual ASR notebook. Hugging Face `transformers` (no ffmpeg / custom attention hooks / DTW code: word timestamps
come from `return_timestamps="word"`), FLEURS is **streamed** (only a few clips), `whisper-small`,
and decoding is compared on WER and speed (greedy vs. beam search).

In [ ]:
for _v in ("backbone", "bundle", "E"):  # free previous case models
    globals().pop(_v, None)
gc.collect()

import itertools
import re

import jiwer
import soundfile as sf
from datasets import Audio as HFAudio
from datasets import load_dataset
from transformers import pipeline

In [ ]:
p2c2 = base.Case(2, 2)
LANG, LANGUAGE, WHISPER, N_CLIPS = "id_id", "indonesian", "openai/whisper-small", 10

samples = p2c2.data / "samples"
if not (samples / "references.csv").exists():
    samples.mkdir(exist_ok=True)

    stream = load_dataset(
        "google/fleurs", LANG, split="test", streaming=True, cache_dir=str(p2c2.data)
    ).cast_column("audio", HFAudio(decode=False))

    rows = []
    for i, ex in enumerate(itertools.islice(stream, N_CLIPS)):
        sf.write(
            samples / f"{i}.wav", base.load_audio(ex["audio"]["bytes"], 16000), 16000
        )
        rows.append({"file": f"{i}.wav", "reference": ex["transcription"]})

    pd.DataFrame(rows).to_csv(samples / "references.csv", index=False)

refs = pd.read_csv(samples / "references.csv")
waves = [base.load_audio(samples / f, 16000) for f in refs.file]
durations = np.array([len(w) / 16000 for w in waves])

print(f"{len(waves)} clips, {durations.sum():.0f}s of audio")

In [ ]:
asr = pipeline(
    "automatic-speech-recognition",
    model=WHISPER,
    model_kwargs={"cache_dir": str(p2c2.data)},
)
print(
    f"{WHISPER}: {sum(p.numel() for p in asr.model.parameters()) / 1e6:.0f}M parameters"
)


def transcribe(wave, beams=1, task="transcribe", timestamps=False):
    out = asr(
        wave,
        generate_kwargs={"language": LANGUAGE, "task": task, "num_beams": beams},
        return_timestamps="word" if timestamps else False,
    )

    return out["text"].strip(), out.get("chunks", [])


norm = lambda t: re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", t.lower())).strip()

Decoding comparison: greedy (`num_beams=1`) vs. beam search (`num_beams=5`)

In [ ]:
runs = {}
for beams in (1, 5):
    texts, secs = [], []
    for w in waves:
        t0 = time.perf_counter()
        texts.append(transcribe(w, beams)[0])
        secs.append(time.perf_counter() - t0)

    hyp, ref = [norm(t) for t in texts], [norm(t) for t in refs.reference]
    runs[beams] = {
        "texts": texts,
        "wer": jiwer.wer(ref, hyp),
        "cer": jiwer.cer(ref, hyp),
        "rtf": sum(secs) / durations.sum(),
        "per_clip": [jiwer.wer(r, h) for r, h in zip(ref, hyp)],
    }

cmp = pd.DataFrame(
    {f"beams={b}": {k: r[k] for k in ("wer", "cer", "rtf")} for b, r in runs.items()}
).T
print(cmp.round(3))

# beam search only wins if it improves WER by more than 0.5 points; otherwise take the faster greedy decoding
best_beams = 5 if runs[1]["wer"] - runs[5]["wer"] > 0.005 else 1

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
pd.DataFrame({f"beams={b}": r["per_clip"] for b, r in runs.items()}).plot.bar(
    ax=ax[0], xlabel="clip", ylabel="WER", title="WER per clip"
)

cmp[["wer", "cer"]].plot.bar(ax=ax[1], rot=0, title="Corpus error rates")

p2c2.save_fig("wer_comparison")

Transcription and translation to English with the selected decoding, plus word-level timestamps

In [ ]:
tr = [transcribe(w, best_beams, "translate")[0] for w in waves]
table = pd.DataFrame(
    {
        "reference": refs.reference,
        "transcription": runs[best_beams]["texts"],
        "translation_en": tr,
    }
)

table.to_csv(p2c2.result / "transcriptions.csv", index=False)
table.head(5)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
for ax, w in zip(axes, waves[:2]):
    text, chunks = transcribe(w, best_beams, timestamps=True)
    ax.plot(np.arange(len(w)) / 16000, w, lw=0.4, c="gray")
    for k, c in enumerate(chunks):
        s, e = c["timestamp"]
        e = e if e is not None else len(w) / 16000
        ax.axvspan(s, e, alpha=0.25, color=f"C{k % 10}")
        ax.text(
            (s + e) / 2,
            w.max() * (0.75 + 0.2 * (k % 2)),
            c["text"].strip(),
            ha="center",
            fontsize=8,
            rotation=60,
        )

    ax.set_ylabel("amplitude")

axes[-1].set_xlabel("time (s)")
p2c2.save_fig("word_timestamps")

# Reference file for the model folder (weights stay in the Hugging Face cache under data/)
p2c2.save_json(
    "reference",
    {
        "model": WHISPER,
        "language": LANGUAGE,
        "num_beams": best_beams,
        "wer": runs[best_beams]["wer"],
        "cer": runs[best_beams]["cer"],
        "real_time_factor_cpu": runs[best_beams]["rtf"],
        "note": "pretrained, not stored here",
    },
    model=True,
)

Inference example: any 16 kHz audio file

In [ ]:
def transcribe_file(path, translate=False):
    return transcribe(
        base.load_audio(path, 16000),
        best_beams,
        "translate" if translate else "transcribe",
    )[0]


clip = samples / "0.wav"
print(
    "transcription:",
    transcribe_file(clip),
    "\ntranslation  :",
    transcribe_file(clip, translate=True),
)

del asr
gc.collect()  # free RAM for the next case
Audio(base.load_audio(clip, 16000), rate=16000)

## Case 3: English text-to-speech
Two pretrained acoustic models, both trained on LJSpeech: **Part A** Coqui FastPitch, **Part B** SpeechBrain FastSpeech2 + HiFi-GAN vocoder.
Both are compared on speed (real-time factor on CPU) and visualised as waveform + mel spectrogram. Weights are cached in `data/project_2/case_3`.

In [ ]:
for _v in "asr":  # free previous case models
    globals().pop(_v, None)
gc.collect()

import os

import librosa
import librosa.display
import soundfile as sf

In [ ]:
p2c3 = base.Case(2, 3)
os.environ["TTS_HOME"] = str(p2c3.data)  # Coqui model cache
TEXT = "Text to speech plays an important role in helping the economic sector to remind customers personally in an organized and effective manner."
outputs = {}  # name -> (waveform, sample_rate, seconds to synthesize)

**Part A: FastPitch (Coqui TTS)**

In [ ]:
from TTS.api import TTS

fastpitch = TTS("tts_models/en/ljspeech/fast_pitch", progress_bar=False)


def say_fastpitch(text):
    t0 = time.perf_counter()
    wav = np.asarray(fastpitch.tts(text), dtype="float32")
    return wav, fastpitch.synthesizer.output_sample_rate, time.perf_counter() - t0


say_fastpitch("warm up")  # exclude one-off initialisation from the timing
outputs["FastPitch"] = say_fastpitch(TEXT)

**Part B: FastSpeech2 + HiFi-GAN (SpeechBrain)**

In [ ]:
from speechbrain.inference.TTS import FastSpeech2
from speechbrain.inference.vocoders import HIFIGAN
from speechbrain.utils.fetching import (
    LocalStrategy,
)  # no symlinks (need admin rights on Windows) and no duplicate cache copy
from transformers import PreTrainedTokenizerBase

In [ ]:
if not hasattr(
    PreTrainedTokenizerBase, "encode_plus"
):  # SpeechBrain's G2P still calls the method removed in transformers 5
    PreTrainedTokenizerBase.encode_plus = lambda self, *a, **k: self(*a, **k)

fs2 = FastSpeech2.from_hparams(
    source="speechbrain/tts-fastspeech2-ljspeech",
    savedir=str(p2c3.data / "fastspeech2"),
    run_opts={"device": "cpu"},
    local_strategy=LocalStrategy.COPY_SKIP_CACHE,
)
hifigan = HIFIGAN.from_hparams(
    source="speechbrain/tts-hifigan-ljspeech",
    savedir=str(p2c3.data / "hifigan"),
    run_opts={"device": "cpu"},
    local_strategy=LocalStrategy.COPY_SKIP_CACHE,
)


def say_fastspeech2(text, pace=1.0, pitch_rate=1.0, energy_rate=1.0):
    t0 = time.perf_counter()
    mel, *_ = fs2.encode_text(
        [text], pace=pace, pitch_rate=pitch_rate, energy_rate=energy_rate
    )

    wav = hifigan.decode_batch(mel).squeeze().cpu().numpy()
    return wav, 22050, time.perf_counter() - t0


say_fastspeech2("warm up")
outputs["FastSpeech2+HiFiGAN"] = say_fastspeech2(TEXT)

Comparison: duration, speed (real-time factor = synthesis time / audio length, lower is faster) and spectrograms

In [ ]:
rows = {
    n: {"audio_s": len(w) / sr, "synth_s": t, "rtf": t / (len(w) / sr)}
    for n, (w, sr, t) in outputs.items()
}

summary = pd.DataFrame(rows).T
print(summary.round(3))
fig, ax = plt.subplots(2, 2, figsize=(12, 5.5))
for j, (n, (w, sr, _)) in enumerate(outputs.items()):
    sf.write(p2c3.result / f"{n.split('+')[0].lower()}.wav", w, sr)
    librosa.display.waveshow(w, sr=sr, ax=ax[0, j])
    ax[0, j].set_title(f"{n}: waveform")
    librosa.display.specshow(
        librosa.power_to_db(
            librosa.feature.melspectrogram(y=w, sr=sr, n_mels=80), ref=np.max
        ),
        sr=sr,
        x_axis="time",
        y_axis="mel",
        ax=ax[1, j],
    )

p2c3.save_fig("tts_comparison")
fastest = summary.rtf.idxmin()

p2c3.save_json(
    "reference",
    {
        "models": {
            "FastPitch": "tts_models/en/ljspeech/fast_pitch",
            "FastSpeech2": "speechbrain/tts-fastspeech2-ljspeech",
            "vocoder": "speechbrain/tts-hifigan-ljspeech",
        },
        "fastest": fastest,
        "metrics": rows,
        "note": "pretrained, not stored here",
    },
    model=True,
)

Inference example: synthesize a new sentence with both models

In [ ]:
sentence = "Speech synthesis turns written text into natural sounding audio."

for name, say in {"FastPitch": say_fastpitch, "FastSpeech2": say_fastspeech2}.items():
    w, sr, t = say(sentence)
    sf.write(p2c3.result / f"example_{name.lower()}.wav", w, sr)
    print(f"{name}: {len(w) / sr:.1f}s of audio in {t:.2f}s")

del fastpitch, fs2, hifigan
gc.collect()  # free RAM for the next case
Audio(w, rate=sr)

## Case 4: Indonesian text-to-speech (VITS)
This case uses Meta's
[`facebook/mms-tts-ind`](https://huggingface.co/facebook/mms-tts-ind): ~145 MB, loaded directly with `transformers`. Weights are cached in `data/project_2/case_4`.

In [ ]:
for _v in ("fastpitch", "fs2", "hifigan"):  # free previous case models
    globals().pop(_v, None)
gc.collect()

import librosa
import librosa.display
import soundfile as sf
from transformers import AutoTokenizer, VitsModel

In [ ]:
p2c4 = base.Case(2, 4)
MODEL = "facebook/mms-tts-ind"
tokenizer = AutoTokenizer.from_pretrained(MODEL, cache_dir=p2c4.data)
vits = VitsModel.from_pretrained(MODEL, cache_dir=p2c4.data).eval()
SR = vits.config.sampling_rate
print(f"{sum(p.numel() for p in vits.parameters()) / 1e6:.0f}M parameters, {SR} Hz")


@torch.no_grad()
def say_indonesian(text, speaking_rate=1.0, seed=base.SEED):
    """text -> (waveform, sample_rate, seconds). VITS samples durations stochastically, so the seed makes it reproducible."""
    torch.manual_seed(seed)
    vits.speaking_rate = speaking_rate
    t0 = time.perf_counter()
    wav = vits(**tokenizer(text.lower(), return_tensors="pt")).waveform[0].numpy()
    return wav, SR, time.perf_counter() - t0

In [ ]:
texts = {
    "stt": "STT adalah teknologi yang memungkinkan kita untuk mengubah ucapan menjadi teks tertulis. Teknologi ini sangat berguna bagi mereka yang ingin mengetik secara lebih cepat dan efisien dengan suara.",
    "tts": "Text to speech berperan penting dalam membantu sektor ekonomi untuk mengingatkan pelanggan secara personal dengan cara yang terorganisir dan efektif.",
}

say_indonesian("halo")  # warm-up, excluded from the timing
fig, ax = plt.subplots(2, len(texts), figsize=(12, 5.5))

rows = {}
for j, (name, text) in enumerate(texts.items()):
    w, sr, t = say_indonesian(text)
    sf.write(p2c4.result / f"{name}.wav", w, sr)
    rows[name] = {"audio_s": len(w) / sr, "synth_s": t, "rtf": t / (len(w) / sr)}

    librosa.display.waveshow(w, sr=sr, ax=ax[0, j])
    ax[0, j].set_title(f"'{name}' sentence: waveform")

    librosa.display.specshow(
        librosa.power_to_db(
            librosa.feature.melspectrogram(y=w, sr=sr, n_mels=80), ref=np.max
        ),
        sr=sr,
        x_axis="time",
        y_axis="mel",
        ax=ax[1, j],
    )

p2c4.save_fig("tts_indonesian")
print(pd.DataFrame(rows).T.round(3))

p2c4.save_json(
    "reference",
    {
        "model": MODEL,
        "architecture": "VITS",
        "sampling_rate": SR,
        "metrics": rows,
        "note": "pretrained, cached in data/project_2/case_4; not stored here",
    },
    model=True,
)

Inference example: any Indonesian sentence

In [ ]:
w, sr, t = say_indonesian("Selamat pagi, semoga harimu menyenangkan.")
sf.write(p2c4.result / "example.wav", w, sr)
print(f"{len(w) / sr:.1f}s of audio in {t:.2f}s")
del vits
gc.collect()  # free RAM for the next case
Audio(w, rate=sr)

## Case 5: Speaker recognition (verification)
SpeechBrain's pretrained **ECAPA-TDNN** speaker encoder (`speechbrain/spkrec-ecapa-voxceleb`, ~80 MB), evaluated on many same-/different-speaker trials
from LibriSpeech (streamed, ~100 MB) with the standard verification metrics (EER, AUC). The decision threshold is tuned on one group of speakers and
tested on a disjoint group.

In [ ]:
for _v in ("vits", "tokenizer"):  # free previous case models
    globals().pop(_v, None)
gc.collect()

import itertools

import soundfile as sf
from datasets import Audio as HFAudio
from datasets import load_dataset
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, roc_curve
from speechbrain.inference.speaker import EncoderClassifier
from speechbrain.utils.fetching import (
    LocalStrategy,  # no symlinks (need admin rights on Windows) and no duplicate cache copy
)

In [ ]:
p2c5 = base.Case(2, 5)
SR, N_SCAN, STEP = 16000, 1000, 4  # scan the first 1000 utterances, keep every 4th
utts = p2c5.data / "utterances"

if not (utts / "index.csv").exists():
    utts.mkdir(exist_ok=True)
    stream = load_dataset(
        "openslr/librispeech_asr",
        "clean",
        split="validation",
        streaming=True,
        cache_dir=str(p2c5.data),
    ).cast_column("audio", HFAudio(decode=False))

    rows = []
    for k, ex in enumerate(itertools.islice(stream, 0, N_SCAN, STEP)):
        sf.write(utts / f"{k}.wav", base.load_audio(ex["audio"]["bytes"], SR), SR)
        rows.append({"file": f"{k}.wav", "speaker": ex["speaker_id"]})

    pd.DataFrame(rows).to_csv(utts / "index.csv", index=False)

idx = pd.read_csv(utts / "index.csv")
print(f"{len(idx)} utterances from {idx.speaker.nunique()} speakers")

In [ ]:
encoder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir=str(p2c5.data / "ecapa"),
    run_opts={"device": "cpu"},
    local_strategy=LocalStrategy.COPY_SKIP_CACHE,
)


@torch.no_grad()
def embed(wave):
    """16 kHz mono waveform -> unit-length 192-d speaker embedding."""
    e = (
        encoder.encode_batch(torch.tensor(wave, dtype=torch.float32)[None])
        .squeeze()
        .numpy()
    )
    return e / np.linalg.norm(e)


t0 = time.perf_counter()
E = np.stack([embed(base.load_audio(utts / f, SR)) for f in idx.file])
print(
    f"embedding: {(time.perf_counter() - t0) / len(idx):.2f}s per utterance, shape {E.shape}"
)

Verification trials: all same-speaker pairs plus an equal number of random different-speaker pairs. Speakers are split into a dev half (choose threshold)
and a test half (report).

In [ ]:
rng = np.random.default_rng(base.SEED)
speakers = rng.permutation(idx.speaker.unique())
group = {s: ("dev" if i % 2 == 0 else "test") for i, s in enumerate(speakers)}


def trials(part):
    ids = np.where(idx.speaker.map(group) == part)[0]
    same = [
        (i, j)
        for i, j in itertools.combinations(ids, 2)
        if idx.speaker[i] == idx.speaker[j]
    ]

    diff = []
    while len(diff) < len(same):
        i, j = rng.choice(ids, 2, replace=False)
        diff.append((i, j)) if idx.speaker[i] != idx.speaker[j] else None

    pairs = np.array(same + diff)
    return np.sum(E[pairs[:, 0]] * E[pairs[:, 1]], 1), np.array(
        [1] * len(same) + [0] * len(diff)
    )  # cosine score, label


def eer_threshold(score, label):
    fpr, tpr, thr = roc_curve(label, score)
    i = np.argmin(np.abs(fpr - (1 - tpr)))
    return thr[i], (fpr[i] + 1 - tpr[i]) / 2


(s_dev, y_dev), (s_te, y_te) = trials("dev"), trials("test")
thr, dev_eer = eer_threshold(s_dev, y_dev)
_, test_eer = eer_threshold(s_te, y_te)
acc = ((s_te >= thr) == y_te).mean()
metrics = {
    "threshold": float(thr),
    "dev_eer": float(dev_eer),
    "test_eer": float(test_eer),
    "test_auc": float(roc_auc_score(y_te, s_te)),
    "test_acc_at_dev_threshold": float(acc),
    "test_trials": len(y_te),
}

print({k: round(v, 4) for k, v in metrics.items()})

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
(
    ax[0].hist(s_te[y_te == 1], 40, alpha=0.6, label="same speaker"),
    ax[0].hist(s_te[y_te == 0], 40, alpha=0.6, label="different speaker"),
)
(
    ax[0].axvline(thr, c="k", ls="--", label=f"threshold {thr:.2f}"),
    ax[0].set(
        title="Cosine score distribution (test speakers)", xlabel="cosine similarity"
    ),
    ax[0].legend(),
)
fpr, tpr, _ = roc_curve(y_te, s_te)
(
    ax[1].plot(fpr, tpr),
    ax[1].plot([0, 1], [0, 1], "k:"),
    ax[1].set(
        title=f"ROC (AUC {metrics['test_auc']:.3f}, EER {test_eer:.1%})",
        xlabel="false positive rate",
        ylabel="true positive rate",
    ),
)
xy = PCA(2).fit_transform(E)
for s in speakers[:12]:
    m = (idx.speaker == s).values
    ax[2].scatter(*xy[m].T, s=14, label=str(s))
ax[2].set(title="Embeddings (PCA, 12 speakers)"), ax[2].legend(fontsize=6, ncol=2)

p2c5.save_fig("speaker_verification")

p2c5.save_json(
    "reference",
    dict(
        model="speechbrain/spkrec-ecapa-voxceleb",
        sample_rate=SR,
        metric="cosine similarity",
        **metrics,
        note="pretrained, cached in data/project_2/case_5; not stored here",
    ),
    model=True,
)

Inference example: are two recordings from the same speaker?

In [ ]:
def verify(path_a, path_b):
    score = float(
        embed(base.load_audio(path_a, SR)) @ embed(base.load_audio(path_b, SR))
    )
    return {"score": round(score, 3), "same_speaker": score >= metrics["threshold"]}


same = idx[idx.speaker == idx.speaker[0]].file.iloc[:2].tolist()
other = idx[idx.speaker != idx.speaker[0]].file.iloc[0]
print("SAME SPEAKER     ", verify(utts / same[0], utts / same[1]))
print("DIFFERENT SPEAKER", verify(utts / same[0], utts / other))
Audio(base.load_audio(utts / same[0], SR), rate=SR)